# 2. Building a property map

A `PropertyMap` is the compartment table: **one row per compartment, one column
per property**, holding `int16` trait codes. Everything downstream — flows,
rates, outputs — will index into that table, so getting its shape and ordering
right is the whole job of this layer.

## Bootstrap from one property

A map has to start somewhere. `from_property` creates one compartment per trait
of a single property.

In [ ]:
from summer4 import Property, PropertyMap

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)

assert pmap.size == 3          # compartments (rows)
assert pmap.n_properties == 1  # properties (columns)
assert pmap.labels() == ("state=S", "state=I", "state=R")

pmap

## Stratify

`stratify(prop)` applies a new axis to every compartment, multiplying the table.
It returns a **new map**; the input is untouched.

In [ ]:
age = Property("age", ("0-4", "5-9", "10+"))

by_age = pmap.stratify(age)

assert by_age.size == 3 * 3
assert pmap.size == 3, "the original map is immutable"

by_age

### Ordering is a cross product in declaration order

New traits vary *fastest*: each existing compartment is expanded into a
contiguous block of its new sub-compartments. That keeps the parent grouping
contiguous in memory, which is what makes later aggregation a slice rather than
a gather.

In [ ]:
assert by_age.labels()[:3] == (
    "state=S_age=0-4",
    "state=S_age=5-9",
    "state=S_age=10+",
)

### Chaining

Stratifications chain, so a model's compartment space is built in one readable
expression.

In [ ]:
vax = Property("vaccination", ("none", "one-dose", "two-dose"))

full = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(vax)
)

assert full.size == 3 * 3 * 3 == 27
assert full.n_properties == 3

## Inspecting a map

Three views are available, in increasing order of structure.

In [ ]:
print(repr(full)[:160], "...\n")
print("label  :", full.labels()[13])
print("dict   :", full.to_dicts()[13])

`to_dicts()` omits properties that do not apply to a compartment, so it is the
view to use when building a human-facing table or a pandas/polars frame.

In [ ]:
rows = full.to_dicts()

assert len(rows) == full.size
assert rows[0] == {"state": "S", "age": "0-4", "vaccination": "none"}
assert all(set(row) == {"state", "age", "vaccination"} for row in rows)

## Retrieving a property from a map

A map registers its properties by name. `get_property` returns the registered
object, which is the safe way to recover an axis inside a function that only
received the map.

In [ ]:
recovered = full.get_property("age")
assert recovered == age
assert recovered.traits == ("0-4", "5-9", "10+")

try:
    full.get_property("sex")
except KeyError as exc:
    print(exc)

## Names must be unique

A property name can appear only once on a map. Applying an axis twice is a
modelling error — if you want a second, independent axis, give it its own name.

In [ ]:
try:
    full.stratify(age)
except ValueError as exc:
    print(exc)

## `Stratification` as a value

`stratify(prop, where=...)` is the convenient form. The same operation is also a
first-class object, which is useful when a stratification has to be stored,
passed around or applied to more than one map.

In [ ]:
from summer4 import Stratification

strat = Stratification(vax)

applied = strat.apply(PropertyMap.from_property(state).stratify(age))
assert applied == full

---

Next: {doc}`03-selectors-and-queries` turns compartments back into index arrays.